# r9p step D: verifier for r9p, dev comparison with r8p, adopt or keep
About 3-5 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, X
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')

R9P = one('runs/v10-r9p/best.pt', 'the output of kinthread-r9p-c-train')
A_DEV = one('private/r8p_dev_final.json', 'the output of kinthread-r9p-a-baseline')
FLAGS = json.load(open(one('public/rescore_dev.json', 'the output of kinthread-r9p-a-baseline')))['adopted_cli']
B1 = json.load(open(one('public/B_r8p_summary.json', 'the output of kinthread-r9p-b1-suites-r8p')))
B2 = json.load(open(one('public/B_r9p_summary.json', 'the output of kinthread-r9p-b2-suites-r9p')))
print('r9p', sha(R9P)[:12], '| flags adopted in step A:', FLAGS or 'none')

In [ ]:
sh(f'python train_verifier.py --ckpt {R9P} --n 1500 --out {PUB}/r9p_verifier.json')
BASE = f'--adaptive 1.0 --hybrid --verifier {PUB}/r9p_verifier.json --cover --split_search --para 2 --k 16'
EXTRA_FLAGS = {'final': '', 'final_plus': f'{FLAGS} --max_new 192 --reserve 8'.strip()}
for name, extra in EXTRA_FLAGS.items():
    sh(f'python -m eval.run_real --ckpt {R9P} --split dev {BASE} {extra} --out {PRIV}/r9p_dev_{name}.json')
    sh(f'python -m eval.compare_runs {A_DEV} {PRIV}/r9p_dev_{name}.json --out {PUB}/compare_{name}.json')

In [ ]:
cmp = {n: json.load(open(f'{PUB}/compare_{n}.json')) for n in EXTRA_FLAGS}
best = max(cmp, key=lambda n: cmp[n]['gain_points'])
suites_ok = (B2['gauntlet_correct'] >= B1['gauntlet_correct'] - 0.01
             and B2['consistency_all_agree'] >= B1['consistency_all_agree'] - 0.01)
decision = {'step': 'D', 'adopt': bool(cmp[best]['adopt'] and suites_ok), 'config': best,
            'extra_flags': EXTRA_FLAGS[best], 'dev': cmp[best], 'all_configs': cmp,
            'gauntlet': {'r8p': B1['gauntlet_correct'], 'r9p': B2['gauntlet_correct']},
            'consistency_all_agree': {'r8p': B1['consistency_all_agree'], 'r9p': B2['consistency_all_agree']},
            'suites_not_worse': suites_ok, 'r9p_sha256': sha(R9P),
            'rule': 'adopt = dev fair gain >= 2 points, wins > losses, McNemar p < 0.05, gauntlet and consistency not lower by more than 1 point'}
json.dump(decision, open(f'{PUB}/decision.json', 'w'), indent=1)
print(json.dumps(decision, indent=1))
print('D DONE: ADOPT r9p' if decision['adopt'] else 'D DONE: KEEP r8p')